# Autograd: How PyTorch Computes Gradients

Three ways to compute derivatives:
1. **By hand** (symbolic) -- exact but tedious, doesn't scale
2. **Finite differences** -- approximate, easy to code, but slow and imprecise
3. **Automatic differentiation (autograd)** -- exact AND automatic

We'll see all three on the same examples, then use PyTorch autograd on a real problem.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12

---
## Part 1: What AutoDiff is NOT -- Finite Differences

The simplest idea: wiggle each input by a tiny amount $h$ and see how the output changes.

One-sided formula:
$$\frac{\partial f}{\partial x_i} \approx \frac{f(x_1, \ldots, x_i + h, \ldots) - f(x_1, \ldots, x_i, \ldots)}{h}$$

Two-sided (central difference) formula:
$$\frac{\partial f}{\partial x_i} \approx \frac{f(x_1, \ldots, x_i + h, \ldots) - f(x_1, \ldots, x_i - h, \ldots)}{2h}$$

In [ ]:
# Example: L = x*y + b  at x=2, y=-3, b=10
# L = 2*(-3) + 10 = 4
# Exact: dL/dx = y = -3, dL/dy = x = 2, dL/db = 1

def L_func(x, y, b):
    return x * y + b

x, y, b = 2.0, -3.0, 10.0
h = 1e-5

dLdx = (L_func(x+h, y, b) - L_func(x-h, y, b)) / (2*h)
dLdy = (L_func(x, y+h, b) - L_func(x, y-h, b)) / (2*h)
dLdb = (L_func(x, y, b+h) - L_func(x, y, b-h)) / (2*h)

print(f'L = x*y + b = {L_func(x,y,b)}')
print(f'dL/dx = {dLdx:.6f}  (exact: {y})')
print(f'dL/dy = {dLdy:.6f}   (exact: {x})')
print(f'dL/db = {dLdb:.6f}   (exact: 1.0)')

### Challenges with Finite Differences

1. **Expensive:** Need a forward pass for EACH variable
2. **Numerically unstable:** Choice of $h$ is tricky -- too large gives a poor approximation, too small gives floating-point errors

In [ ]:
# Cost: need 2 forward passes PER parameter
n_params = [3, 100, 1_000_000, 175_000_000_000]
names = ['Toy example', 'Small model', 'ResNet-50', 'GPT-3']

print('Cost of finite differences:')
print(f'{"Model":<20} {"Parameters":>15} {"Forward passes":>20}')
print('-' * 57)
for name, n in zip(names, n_params):
    print(f'{name:<20} {n:>15,} {2*n:>20,}')

print()
print('GPT-3 would need 350 BILLION forward passes for one gradient step!')
print('Autograd does it in ONE backward pass.')

---
## Part 2: Computational Graphs & Backprop

The key idea behind autograd:
- **Nodes:** Operations (+, ×, exp, ...)
- **Edges:** Variables / Tensors (data dependencies)

Example: $L = x \cdot y + b$

```
x (2) ──→ [×] ──→ mul (-6) ──→ [+] ──→ L (4)
y (-3) ─↗                        ↑
                           b (10) ┘
```

### Backprop rule

For a node $z = f(x, y)$ in a larger computation ending at loss $J$:

$$\frac{dJ}{dx} = \underbrace{\frac{dJ}{dz}}_{\text{upstream grad}} \cdot \underbrace{\frac{dz}{dx}}_{\text{local grad}}$$

$$\boxed{\text{downstream gradient} = \text{upstream gradient} \times \text{local gradient}}$$

### Manual backward pass: $L = x \cdot y + b$

In [ ]:
# Manual backward pass for L = x*y + b
# x=2, y=-3, b=10  →  mul=-6, L=4

# Step 1: dL/dL = 1
dL_dL = 1.0
print(f'dL/dL = {dL_dL}')

# Step 2: L = mul + b  (addition: local grads are both 1)
dL_dmul = dL_dL * 1.0  # upstream × local
dL_db   = dL_dL * 1.0
print(f'dL/d(mul) = {dL_dL} × 1 = {dL_dmul}')
print(f'dL/db     = {dL_dL} × 1 = {dL_db}')

# Step 3: mul = x * y  (multiplication: local grads are y and x)
x_val, y_val = 2.0, -3.0
dL_dx = dL_dmul * y_val  # upstream × local
dL_dy = dL_dmul * x_val
print(f'dL/dx     = {dL_dmul} × y = {dL_dmul} × {y_val} = {dL_dx}')
print(f'dL/dy     = {dL_dmul} × x = {dL_dmul} × {x_val} = {dL_dy}')

print()
print('Summary:')
print(f'  dL/dx = {dL_dx}   (increase x by 1 → L decreases by 3)')
print(f'  dL/dy = {dL_dy}    (increase y by 1 → L increases by 2)')
print(f'  dL/db = {dL_db}    (increase b by 1 → L increases by 1)')

---
## Part 3: PyTorch Does This Automatically

Set `requires_grad=True` and PyTorch tracks the computation graph. One call to `.backward()` computes all gradients.

In [ ]:
# Same example: L = x*y + b
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(-3.0, requires_grad=True)
b = torch.tensor(10.0, requires_grad=True)

mul = x * y
L = mul + b

print(f'L = {L.item()}')
print(f'L.grad_fn = {L.grad_fn}')  # PyTorch knows how L was computed

# One call computes ALL gradients!
L.backward()

print(f'\ndL/dx = {x.grad.item()}   (exact: -3.0)')
print(f'dL/dy = {y.grad.item()}    (exact: 2.0)')
print(f'dL/db = {b.grad.item()}    (exact: 1.0)')
print('\nExact. No approximation. No manual math.')

### Side-by-side: all three methods

In [ ]:
print(f'{"":<25} {"dL/dx":>10} {"dL/dy":>10} {"dL/db":>10} {"Exact?":>8}')
print('=' * 65)
print(f'{"By hand":<25} {-3.0:>10.4f} {2.0:>10.4f} {1.0:>10.4f} {"Yes":>8}')
print(f'{"Finite differences":<25} {dLdx:>10.4f} {dLdy:>10.4f} {dLdb:>10.4f} {"No":>8}')
print(f'{"PyTorch autograd":<25} {x.grad.item():>10.4f} {y.grad.item():>10.4f} {b.grad.item():>10.4f} {"Yes":>8}')

---
## Part 4: Another Example -- $y = wx + b$, $\text{loss} = y^2$

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(2.0)  # input, no grad needed
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b       # y = 3*2 + 1 = 7
loss = y ** 2        # loss = 49

loss.backward()

print(f'y = w*x + b = {y.item()}')
print(f'loss = y**2 = {loss.item()}')
print(f'\ndloss/dw = {w.grad.item()}   (= 2*y*x = 2*7*2 = 28)')
print(f'dloss/db = {b.grad.item()}   (= 2*y*1 = 2*7 = 14)')

In [ ]:
# Gradient descent update
lr = 0.01
with torch.no_grad():
    w.data -= lr * w.grad
    b.data -= lr * b.grad

print(f'After one gradient descent step (lr={lr}):')
print(f'  w: 3.0 → {w.data.item():.4f}')
print(f'  b: 1.0 → {b.data.item():.4f}')

# IMPORTANT: zero the gradients!
w.grad.zero_()
b.grad.zero_()
print()
print('Always call zero_grad() before the next backward pass!')

In [ ]:
# Why zero_grad matters: gradients ACCUMULATE
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
y.backward()
print(f'After 1st backward: x.grad = {x.grad.item()}  (should be 2x = 6)')

y = x ** 2
y.backward()
print(f'After 2nd backward: x.grad = {x.grad.item()}  (6 + 6 = 12, NOT 6!)')

x.grad.zero_()
y = x ** 2
y.backward()
print(f'After zero_grad():  x.grad = {x.grad.item()}  (correct!)')

print()
print('Forgetting zero_grad() is the #1 PyTorch beginner bug.')

---
## Part 5: Logistic Regression Loss -- Step by Step

$$\hat{y} = \sigma(\theta_0 + \theta_1 x_1 + \theta_2 x_2) = \frac{1}{1 + e^{-(\theta_0 + \theta_1 x_1 + \theta_2 x_2)}}$$

For $y = 1$:
$$\text{Loss} = -\log(\hat{y})$$

### The computation graph (elementary ops)

$$\theta_1, x_1 \xrightarrow{\times} f_1 \quad\quad \theta_2, x_2 \xrightarrow{\times} f_2$$
$$f_1, f_2 \xrightarrow{+} f_3 \xrightarrow{+\theta_0} f_4 \xrightarrow{\times(-1)} f_5 \xrightarrow{\exp} f_6 \xrightarrow{+1} f_7 \xrightarrow{1/x} f_8 \xrightarrow{\log} f_9 \xrightarrow{\times(-1)} L$$

Let's trace through with $\theta_1 = 1, x_1 = 1, \theta_2 = 2, x_2 = 2, \theta_0 = 1$.

In [ ]:
# Forward pass: step by step through each elementary operation
theta1, x1_val = 1.0, 1.0
theta2, x2_val = 2.0, 2.0
theta0 = 1.0

f1 = theta1 * x1_val                  # θ₁ × x₁
f2 = theta2 * x2_val                  # θ₂ × x₂
f3 = f1 + f2                           # +
f4 = f3 + theta0                       # + θ₀
f5 = f4 * (-1)                         # × (-1)
f6 = math.exp(f5)                      # exp
f7 = f6 + 1                            # + 1
f8 = 1 / f7                            # 1/x  → this is σ(z)
f9 = math.log(f8)                      # log
L  = f9 * (-1)                         # × (-1) → Loss

print('Forward pass through elementary ops:')
print(f'  f1 = θ₁ × x₁       = {f1}')
print(f'  f2 = θ₂ × x₂       = {f2}')
print(f'  f3 = f1 + f2        = {f3}')
print(f'  f4 = f3 + θ₀        = {f4}')
print(f'  f5 = f4 × (-1)      = {f5}')
print(f'  f6 = exp(f5)        = {f6:.6f}')
print(f'  f7 = f6 + 1         = {f7:.6f}')
print(f'  f8 = 1/f7 = σ(6)    = {f8:.6f}')
print(f'  f9 = log(f8)        = {f9:.6f}')
print(f'  L  = -f9            = {L:.6f}')

In [ ]:
# PyTorch autograd does the backward pass through all these ops automatically
t1 = torch.tensor(1.0, requires_grad=True)
t2 = torch.tensor(2.0, requires_grad=True)
t0 = torch.tensor(1.0, requires_grad=True)
tx1 = torch.tensor(1.0)
tx2 = torch.tensor(2.0)

z = t1 * tx1 + t2 * tx2 + t0
loss = -torch.log(torch.sigmoid(z))

loss.backward()

print(f'Loss = {loss.item():.6f}')
print(f'\nGradients (computed automatically):')
print(f'  dL/dθ₁ = {t1.grad.item():.6f}')
print(f'  dL/dθ₂ = {t2.grad.item():.6f}')
print(f'  dL/dθ₀ = {t0.grad.item():.6f}')
print(f'\nPyTorch walked the graph backward through all 10 elementary ops!')
print(f'We would need {3 * 2} forward passes for finite differences (2 per parameter).')
print(f'Autograd needed just 1 backward pass.')

In [ ]:
# Verify with finite differences
h = 1e-7

def logreg_loss(t1, t2, t0, x1, x2):
    z = t1*x1 + t2*x2 + t0
    return -math.log(1 / (1 + math.exp(-z)))

dLdt1_fd = (logreg_loss(1+h,2,1,1,2) - logreg_loss(1-h,2,1,1,2)) / (2*h)
dLdt2_fd = (logreg_loss(1,2+h,1,1,2) - logreg_loss(1,2-h,1,1,2)) / (2*h)
dLdt0_fd = (logreg_loss(1,2,1+h,1,2) - logreg_loss(1,2,1-h,1,2)) / (2*h)

print(f'{"":<25} {"dL/dθ₁":>12} {"dL/dθ₂":>12} {"dL/dθ₀":>12}')
print('=' * 63)
print(f'{"Finite differences":<25} {dLdt1_fd:>12.6f} {dLdt2_fd:>12.6f} {dLdt0_fd:>12.6f}')
print(f'{"PyTorch autograd":<25} {t1.grad.item():>12.6f} {t2.grad.item():>12.6f} {t0.grad.item():>12.6f}')
print()
print('Same answers -- but autograd is exact and scales to billions of parameters.')

---
## Part 6: Autograd on a Small Neural Network

In [ ]:
import time

# Small network: 100 → 50 → 10
torch.manual_seed(42)
model = torch.nn.Sequential(
    torch.nn.Linear(100, 50),
    torch.nn.ReLU(),
    torch.nn.Linear(50, 10)
)

x_input = torch.randn(1, 100)
target = torch.zeros(1, dtype=torch.long)
loss_fn = torch.nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model has {n_params:,} parameters')

# Autograd: one backward pass
start = time.time()
for _ in range(100):
    model.zero_grad()
    loss = loss_fn(model(x_input), target)
    loss.backward()
t_auto = (time.time() - start) / 100

print(f'\nAutograd: {t_auto*1000:.2f} ms per gradient computation')
print(f'Finite differences would need {2 * n_params:,} forward passes per step')
print(f'\nThis is why deep learning is possible!')

---
## Part 7: The Pattern You'll Always Use

```python
# 1. Forward pass (PyTorch records the graph)
output = model(input)
loss = loss_fn(output, target)

# 2. Backward pass (autograd computes ALL gradients)
loss.backward()

# 3. Update parameters
optimizer.step()
optimizer.zero_grad()
```

`loss.backward()` walks the graph in reverse, applying the chain rule at every node.

---
## Summary

| Method | Exact? | Cost | Scales? |
|--------|--------|------|--------|
| By hand (symbolic) | Yes | Human time | No way |
| Finite differences | No | 2N forward passes | Very slow |
| **Autograd** | **Yes** | **1 backward pass** | **Yes!** |

**How autograd works:**
1. Forward pass records a computation graph
2. Each operation knows its own local gradient
3. Backward pass walks the graph in reverse: **downstream = upstream × local**
4. Result: exact gradients for ALL parameters in one pass

**This is why deep learning is possible.** Without autograd, training a network with millions of parameters would require millions of forward passes per gradient step.